# 03 — Chronological Empirical Design

The empirical blocks are declared before any probabilistic model is
fitted. Random splitting is prohibited and settlement date is the
uncertainty unit.

In [1]:
from pathlib import Path
import json

import pandas as pd
import yaml

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "chronology_policy.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

panel = pd.read_csv(
    ROOT
    / "data/processed/"
    "02_chronological_design_panel.csv"
)

blocks = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "03_chronology_block_summary.csv"
)

folds = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "03_development_fold_summary.csv"
)

policy = yaml.safe_load(
    (
        ROOT
        / "config/"
        "chronology_policy.yaml"
    ).read_text(encoding="utf-8")
)

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "03_chronology_manifest.json"
    ).read_text(encoding="utf-8")
)

print("Status:", policy["status"])
print(
    "Model fitting permitted:",
    policy["model_fitting_permitted"],
)
print()
print(blocks.to_string(index=False))

Status: CHRONOLOGY_ASSIGNED
Model fitting permitted: True

      chronology_block  dates start_date   end_date  date_rule_rows
       warmup_training     24 2026-03-16 2026-04-11              64
development_validation     38 2026-04-12 2026-05-21             152
               holdout     10 2026-05-22 2026-05-31              40
         external_test     30 2026-06-01 2026-06-30             119


## Expanding validation

The warm-up block forms the initial training sample. Development dates
are divided into four contiguous validation folds.

For every fold, all training dates precede the validation dates. Model
families are compared using date-grouped out-of-fold CRPS.

In [2]:
assert len(folds) == 4
assert folds["date_sets_disjoint"].all()
assert folds["training_precedes_validation"].all()

print(folds.to_string(index=False))

 fold  training_dates training_start training_end  validation_dates validation_start validation_end  date_sets_disjoint  training_precedes_validation
    1              24     2026-03-16   2026-04-11                10       2026-04-12     2026-04-21                True                          True
    2              34     2026-03-16   2026-04-21                10       2026-04-22     2026-05-01                True                          True
    3              44     2026-03-16   2026-05-01                 9       2026-05-02     2026-05-10                True                          True
    4              53     2026-03-16   2026-05-10                 9       2026-05-11     2026-05-21                True                          True


## Locked evaluation periods

Holdout and external outcomes cannot affect the model family,
hyperparameters, calibration choice or trading rule.

The model fitted before the holdout is transferred to June without
refitting after observing holdout outcomes.

In [3]:
locked = panel[
    "chronology_block"
].isin(
    ["holdout", "external_test"]
)

assert not panel.loc[
    locked,
    "outcome_may_influence_model_choice",
].any()

assert not panel[
    "used_to_refit_before_external_test"
].any()

assert manifest["holdout_locked"] is True
assert manifest["external_test_locked"] is True
assert manifest["refit_before_external_test"] is False

print(
    "Holdout and external outcomes excluded from selection:",
    True,
)

Holdout and external outcomes excluded from selection: True


## Later observations

July and August observations may be appended as an additional temporal
extension. They cannot retroactively alter any model or decision rule
selected from the declared development period.